<a href="https://colab.research.google.com/github/beenishtech/Library-Management-System/blob/main/imdb_sentiment_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Library Load

In [2]:
import pandas as pd # Import pandas library to work with dataframes
from sklearn.model_selection import train_test_split #Import function to split data into train and test sets
from tensorflow.keras.preprocessing.text import Tokenizer # Import tool to convert words into unique numbers
from tensorflow.keras.preprocessing.sequence import pad_sequences # Import tool to make all text lengths equal
from tensorflow.keras.models import Sequential # Import Sequential to build the model layer by layer
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout # Import the required deep learning layers

#Data Read

In [1]:
url = "https://raw.githubusercontent.com/Ankit152/IMDB-Sentiment-Analysis/master/IMDB-Dataset.csv" # Load the IMDB movie reviews dataset directly from a web URL

df = pd.read_csv(url) # Read the CSV file and save it inside a pandas dataframe (df)

print(df.head()) # Data view

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


#Data Preprocessing

In [3]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})# Convert text labels ('positive' to 1 and 'negative' to 0) for the computer to understand

# X (Reviews) and y (Sentiments) seprated
X = df['review'].values # Inputs: Extract all text reviews into a numpy array
y = df['sentiment'].values# Outputs: Extract all 0 and 1 sentiment labels into another array

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # Split the data: 80% for training the model and 20% for testing its accuracy

#Tokenization aur Padding

In [4]:
max_words = 10000 # Only keep the top 10,000 most frequent used words in the dataset
max_len = 200 # Fix the maximum length of every movie review to 200 words


tokenizer = Tokenizer(num_words=max_words)# Create a tokenizer object with our word limit
tokenizer.fit_on_texts(X_train)# Teach the tokenizer all the unique words present in the training data

#Text convert into numbers:
X_train_seq = tokenizer.texts_to_sequences(X_train)# Convert english words in training data into sequences of numbers
X_test_seq = tokenizer.texts_to_sequences(X_test)# Convert english words in testing data into sequences of numbers


# Padding:
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)# Pad short reviews with 0s and cut long reviews to exactly 200 words
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)# Do the exact same 200-word padding for the test data

#Create the LSTM Model Architecture



In [6]:
model = Sequential()# Initialize an empty sequential model container

# Embedding Layer: Helps the model learn the meaning and relationship between different words
model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_len))

# #LSTM Layer: 64 memory(units) cells to capture the sequence and flow of text. Dropout helps prevent overfitting  OR remeber the 64units memory
model.add(LSTM(64, dropout=0.2, recurrent_dropout=0.2))

#Output Layer: A dense layer with sigmoid function to output a probabilty score between 0 and 1
model.add(Dense(1, activation='sigmoid'))

# Compile Model: Use 'adam' optimizer and 'binary_crossentropy' loss since we have only two categories (0 and 1)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print(model.summary()) # Display the structural summary of the built model on the screen

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


#Model Train

In [7]:
# Start training the model and save its complete training progress (accuracy/loss) in the 'history' variable
history = model.fit(
    X_train_pad, y_train, # Pass the padded training text data and their correct target answers
    epochs=3, # The model will go through the entire dataset 3 times (3 training rounds)
    batch_size=64, # Pass the data in small batches of 64 reviews at a time for efficiency
    validation_data=(X_test_pad, y_test)# Evaluate the model on test data after every single round
)

Epoch 1/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 208s 326ms/step - accuracy: 0.7909 - loss: 0.4527 - val_accuracy: 0.8487 - val_loss: 0.3559
Epoch 2/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 263s 328ms/step - accuracy: 0.8632 - loss: 0.3351 - val_accuracy: 0.8632 - val_loss: 0.3340
Epoch 3/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 209s 335ms/step - accuracy: 0.8833 - loss: 0.2939 - val_accuracy: 0.8658 - val_loss: 0.3196


#Review Test

In [8]:
def predict_my_review(new_review):# Define a custom function to test any random movie review text
    # Review's preprocessing
    seq = tokenizer.texts_to_sequences([new_review])# Convert the incoming text review into a number sequence
    padded = pad_sequences(seq, maxlen=max_len)# Pad the sequence to match the fixed length of 200 words

    # Prediction
    prediction = model.predict(padded)[0][0]# Get the model's output probability for being positive

    if prediction > 0.5:# If the probability is greater than 50%, classify it as positive
        print(f"Review: '{new_review}'POSITIVE ({prediction:.2f})")# Print positive message with score
    else:# If the probability is 50% or less, classify it as negative
        print(f"Review: '{new_review}'NEGATIVE ({prediction:.2f})")# Print negative message with score

# Call the function with sample reviews to verify how well the trained model performs
predict_my_review("The movie was absolutely fantastic and brilliant! I loved it.")
predict_my_review("Total waste of time. The acting was terrible and story was boring.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 395ms/step
Review: 'The movie was absolutely fantastic and brilliant! I loved it.'POSITIVE (0.96)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
Review: 'Total waste of time. The acting was terrible and story was boring.'NEGATIVE (0.01)


#predictions

In [10]:
predict_my_review("The plot was weak, but the visual effects were stunning. Overall average.")
predict_my_review("An absolute masterpiece! Best acting I have ever seen.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
Review: 'The plot was weak, but the visual effects were stunning. Overall average.'NEGATIVE (0.46)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Review: 'An absolute masterpiece! Best acting I have ever seen.'POSITIVE (0.79)
